# Forecasting a London household's next 24 hours of electricity use with an LSTM and a GRU

**Patrick Gilliland**, AI Mastery Capstone: Deep Learning Systems

This is a sequential or time-series task using a Recurrent Neural Network (RNN). From one household's previous seven days of half-hourly readings, the notebook forecasts that household's next 24 hours (48 half-hourly readings, in kWh). Two recurrent models make the forecast, and they differ only in the layer type: the baseline model (the LSTM) and the experimental model (the GRU). The data is the Low Carbon London smart meter data (UK Power Networks), 1,000 households from 2013. The report ships as `Deep_Learning_Systems_Analysis_Report.pdf` and, with the same content, as `module_summary.pdf`.

**Setup notes**

| Item | This notebook |
|---|---|
| Rig | NVIDIA GeForce RTX 5090, Linux under WSL2 |
| Python | 3.12.3 |
| PyTorch | 2.14.1 from PyPI |
| CUDA | 13.0 (the runtime PyTorch ships with), cuDNN 9.24.0 |
| Training time on the GPU, last full run | [TO FILL from the Phase 2 full run] |
| Estimated training time on a CPU | [TO FILL from the Phase 2 full run] |

Both times come from the last full run of this notebook, and the code prints them in Task 5's timing table. With no GPU visible, the same code runs on the CPU.

**Setup.** The first code cell sets two environment variables before PyTorch loads, seeds every random number generator, fixes the settings that make a rerun repeat its numbers, caps the GPU's memory when there is a GPU, and defines the training watchdog that Task 3 uses.

In [1]:
import math
import os
import platform
import random
import statistics

# A value set outside the notebook wins, so CUDA_VISIBLE_DEVICES set to an empty string hides
# the GPU for a CPU run. The cuBLAS setting is the one PyTorch's nn.LSTM page gives for
# repeatable recurrent layers on a GPU.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:2")

import numpy as np  # noqa: E402
import torch  # noqa: E402

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic cuDNN kernels and no TF32, so a rerun gives the same numbers and the arithmetic
# stays float32.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

on_gpu = torch.cuda.is_available()
if on_gpu:
    # Half the card at most, by this rig's GPU rules.
    torch.cuda.set_per_process_memory_fraction(0.5)


def watchdog(epoch_seconds, loss, elapsed_seconds, on_gpu, factor=3.0, limit_seconds=7200.0):
    """Return why a training run should stop, or None to carry on.

    epoch_seconds lists each finished epoch's seconds, the latest last. A NaN or infinite loss
    stops a run on any device. On the GPU, an epoch slower than factor times the median of the
    epochs before it, or a run past limit_seconds, also stops it. On a CPU the run goes to the
    end, because a grader's CPU run is meant to finish.
    """
    if not math.isfinite(loss):
        return f"loss is {loss}"
    if not on_gpu:
        return None
    if elapsed_seconds > limit_seconds:
        return f"the run passed {limit_seconds / 3600:g} hours"
    if len(epoch_seconds) > 1:
        line = factor * statistics.median(epoch_seconds[:-1])
        if epoch_seconds[-1] > line:
            return f"the epoch took {epoch_seconds[-1]:.1f} s, over {line:.1f} s"
    return None


print(f"Python {platform.python_version()}, PyTorch {torch.__version__}, NumPy {np.__version__}")
print(f"Platform: {platform.platform()}")
print(f"Seed: {SEED}")
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']!r}, "
      f"CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']!r}")
if on_gpu:
    cudnn = torch.backends.cudnn.version()
    print(f"Device: cuda, {torch.cuda.get_device_name(0)}, CUDA {torch.version.cuda}, "
          f"cuDNN {cudnn // 10000}.{cudnn % 10000 // 100}.{cudnn % 100}")
    print("GPU memory cap: half the card")
else:
    print("Device: cpu, no GPU visible, so the watchdog lets a run go to the end")

Python 3.12.3, PyTorch 2.14.1+cu130, NumPy 2.5.4
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
Seed: 42
CUDA_VISIBLE_DEVICES='0', CUBLAS_WORKSPACE_CONFIG=':4096:2'
Device: cuda, NVIDIA GeForce RTX 5090, CUDA 13.0, cuDNN 9.24.0
GPU memory cap: half the card


**Environment check.** The brief's own validation code, as given, reading this project's data file. In one cell only the last line's value is shown, so `device` on its own line prints nothing here; the setup cell above prints the device.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device
df = pd.read_csv("lcl_2013_1000_households.csv.gz")
df.head()

,LCLid,tariff_group,date,price_signal,r00,r01,r02,r03,r04,r05,...,r38,r39,r40,r41,r42,r43,r44,r45,r46,r47
0,MAC000006,Std,2013-01-01,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.044,0.044,0.043,0.044,0.030,0.0,0.0,0.0,0.0,0.0
1,MAC000006,Std,2013-01-02,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.044,0.043,0.043,0.044,0.029,0.0,0.0,0.0,0.0,0.0
2,MAC000006,Std,2013-01-03,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.043,0.043,0.043,0.043,0.024,0.0,0.0,0.0,0.0,0.0
3,MAC000006,Std,2013-01-04,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.043,0.044,0.043,0.043,0.030,0.0,0.0,0.0,0.0,0.0
4,MAC000006,Std,2013-01-05,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.043,0.043,0.043,0.043,0.030,0.0,0.0,0.0,0.0,0.0


## Task 1: Define the Problem and Choose Your Dataset

## Task 2: Load and Inspect Your Dataset

## Task 3: Implement a Baseline Deep Learning Model

## Task 4: Conduct an Experimental Comparison

## Task 5: Evaluate and Compare Model Performance

## Summary